# 2 · Hyperparameter search

Finds good settings for the three model families on the annotated series 1.

* The window features are computed **once** per sausage (`ml_core.training.load_windows`). Windowing is
  stateless, so this gives the same result as windowing inside every fold, but much faster.
* Successive halving grid search (`HalvingGridSearchCV`) with **leave-one-sausage-out** cross-validation,
  scored by **balanced accuracy**, because the phases have very different lengths.
* The winners are written to `data/models/hyperparameters.json`. Notebook 3 reads them from there.
* `FEATURE_SET` selects which sensor signals the models may use (`FEATURE_SETS` in `ml_core/training.py`). The default,
  `temperature_difference`, uses the probe temperatures relative to the air instead of absolute temperatures: in the only
  annotated run (spring → summer) the absolute temperature follows the season (see notebook 04). Search with the same
  feature set you train with in notebook 3.
* Search space: `PARAM_GRIDS` in `ml_core/training.py`. `QUICK = True` runs tiny grids, for testing the notebook only.

Runtime on the full data: tens of minutes on CPU. If RAPIDS is installed (GPU), the next cell switches it on.

In [1]:
# Optional GPU acceleration. Must run before scikit-learn is imported.
try:
    import cuml
    cuml.accel.install()
    GPU = True
except Exception as e:  # not installed or no GPU: plain scikit-learn on CPU
    GPU = False
print("RAPIDS cuML acceleration:", GPU)

RAPIDS cuML acceleration: False


In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import pandas as pd
from ml_core import data
from ml_core.training import (ALL_MODELS, DEFAULT_CONTEXT, DEFAULT_FEATURE_SET, DEFAULT_STRIDE, DEFAULT_WINDOW_SIZE,
                              MODEL_NAMES, PARAM_GRIDS, QUICK_GRIDS, load_windows, save_hyperparameters,
                              search_hyperparameters)
pd.set_option("display.max_colwidth", 200)

In [3]:
CONVERTED_DIR = data.CONVERTED_DIR
MODELS_DIR = data.MODELS_DIR
WINDOW_SIZE, STRIDE = DEFAULT_WINDOW_SIZE, DEFAULT_STRIDE   # uplinks per window, step between windows
CONTEXT = DEFAULT_CONTEXT           # long-term context horizons, e.g. ("24h", "7d"); compared in notebook 05
FEATURE_SET = DEFAULT_FEATURE_SET   # see FEATURE_SETS in ml_core/training.py
MODELS = list(ALL_MODELS)           # e.g. ["hgb"] to search one model again; the others' results are kept
QUICK = False
N_JOBS = 1 if GPU else -1

In [4]:
windows = load_windows(CONVERTED_DIR, window_size=WINDOW_SIZE, stride=STRIDE, context=CONTEXT).select(FEATURE_SET)
print(f"Feature set {FEATURE_SET}: {len(windows.columns)} features per window")
windows.summary()

series_1_sausage_1: 41,462 rows → 19,388 windows


series_1_sausage_2: 41,479 rows → 19,395 windows
series_1_sausage_3: 41,468 rows → 19,392 windows
series_1_sausage_4: 41,472 rows → 19,394 windows
series_1_sausage_5: 41,465 rows → 19,388 windows
Feature set temperature_difference_now: 36 features per window


,recording,rows,windows,from,to,Umroetung,Reifung Phase 1,Reifung Phase 2
0,series_1_sausage_1,41462,19388,2023-04-06 00:10:12.000000+00:00,2023-08-02 23:56:34.268438+00:00,5965,8116,5307
1,series_1_sausage_2,41479,19395,2023-04-06 00:11:14.000000+00:00,2023-08-02 23:43:26.013334+00:00,5969,8120,5306
2,series_1_sausage_3,41468,19392,2023-04-06 00:13:24.774014+00:00,2023-08-02 23:58:00.000000+00:00,5969,8118,5305
3,series_1_sausage_4,41472,19394,2023-04-06 00:05:37.000000+00:00,2023-08-02 23:55:25.481812+00:00,5970,8120,5304
4,series_1_sausage_5,41465,19388,2023-04-06 00:21:09.446387+00:00,2023-08-02 23:49:39.000000+00:00,5967,8117,5304


In [5]:
result = search_hyperparameters(windows, keys=MODELS, grids=QUICK_GRIDS if QUICK else PARAM_GRIDS, n_jobs=N_JOBS)


=== Logistic Regression ===  (feature set: temperature_difference_now)


n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 1939
max_resources_: 96957
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 27
n_resources: 1939
Fitting 5 folds for each of 27 candidates, totalling 135 fits


----------
iter: 1
n_candidates: 7
n_resources: 7756
Fitting 5 folds for each of 7 candidates, totalling 35 fits


----------
iter: 2
n_candidates: 2
n_resources: 31024
Fitting 5 folds for each of 2 candidates, totalling 10 fits


best balanced_accuracy = 0.768 in 3.2 min: {'C': 10, 'class_weight': 'balanced', 'l1_ratio': 0, 'max_iter': 1000, 'solver': 'lbfgs'}

=== Random Forest ===  (feature set: temperature_difference_now)
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 1939
max_resources_: 96957
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 32
n_resources: 1939
Fitting 5 folds for each of 32 candidates, totalling 160 fits


----------
iter: 1
n_candidates: 8
n_resources: 7756
Fitting 5 folds for each of 8 candidates, totalling 40 fits


... (4 outputs left out)

best balanced_accuracy = 0.716 in 3.3 min: {'activation': 'tanh', 'alpha': 0.001, 'batch_size': 32, 'early_stopping': True, 'hidden_layer_sizes': [64, 32], 'learning_rate_init': 0.001, 'max_iter': 500, 'validation_fraction': 0.15}

=== Gradient Boosting ===  (feature set: temperature_difference_now)
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 1939
max_resources_: 96957
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 48
n_resources: 1939
Fitting 5 folds for each of 48 candidates, totalling 240 fits


----------
iter: 1
n_candidates: 12
n_resources: 7756
Fitting 5 folds for each of 12 candidates, totalling 60 fits


----------
iter: 2
n_candidates: 3
n_resources: 31024
Fitting 5 folds for each of 3 candidates, totalling 15 fits


best balanced_accuracy = 0.786 in 3.0 min: {'class_weight': None, 'early_stopping': True, 'l2_regularization': 1.0, 'learning_rate': 0.05, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'validation_fraction': 0.1}


## Best candidates per model (last halving round)

In [6]:
for key, r in result.items():
    print(f"{MODEL_NAMES[key]}: best balanced accuracy {r['score']:.3f}")
    display(r["results"].head(5))

Logistic Regression: best balanced accuracy 0.768


,C,class_weight,l1_ratio,max_iter,solver,balanced_accuracy,std
0,10,balanced,0,1000,lbfgs,0.768,0.044
1,1,balanced,1,1000,liblinear,0.768,0.045


Random Forest: best balanced accuracy 0.770


,bootstrap,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators,balanced_accuracy,std
0,True,None,gini,10,sqrt,1,2,500,0.77,0.082
1,True,None,gini,10,sqrt,1,2,200,0.77,0.082


MLP (neural net): best balanced accuracy 0.716


,activation,alpha,batch_size,early_stopping,hidden_layer_sizes,learning_rate_init,max_iter,validation_fraction,balanced_accuracy,std
0,tanh,0.0010,32,True,"(64, 32)",0.001,500,0.15,0.716,0.080
1,tanh,0.0001,32,True,"(32, 16)",0.001,500,0.15,0.714,0.074


Gradient Boosting: best balanced accuracy 0.786


,class_weight,early_stopping,l2_regularization,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,validation_fraction,balanced_accuracy,std
0,None,True,1.0,0.05,300,31,20,0.1,0.786,0.100
1,None,True,1.0,0.10,300,31,20,0.1,0.786,0.100
2,None,True,1.0,0.05,300,63,20,0.1,0.785,0.099


In [7]:
path = save_hyperparameters(result, windows, MODELS_DIR)
print("Saved", path)

Saved F:\_Data\Uni\MA_Project\masterproject-lorawan-demonstrator\data\models\hyperparameters.json
